In [ ]:
#Imports
from model.ccagame import calc_eigengame_eigenvectors
import numpy as np
import jax.numpy as jnp
import jax.scipy as jsp
from jax import random
from sklearn.cross_decomposition import CCA
from jax import jit, grad
from functools import partial

In [ ]:
#Parameters
random_state = 0
n=50
p = 8
q = 9
latent_dims = 5
max_iter = 200
riemannian_projection = False
lr = 5e-1


In [ ]:
#Data Generation
key = random.PRNGKey(random_state)
key, subkey = random.split(key)
X = random.normal(key, (n, p))
Y = random.normal(subkey, (n, q))

Xnp = np.array(X)
Ynp = np.array(Y)


In [ ]:
#Model
cca = CCA(n_components=p, scale=False).fit(Xnp, Ynp)
ccax, ccay = cca.transform(Xnp, Ynp)
cca_corr = np.diag(np.corrcoef(ccax, ccay, rowvar=False)[p:, :p])
p, U1np, V1np = calc_numpy_eig(X, Y, n=latent_dims)
print("\n Eigenvalues calculated using numpy are :\n", p)
U1, V1 = calc_eigengame_eigenvectors(X, Y, latent_dims, lr=lr, iterations=max_iter,
                                     riemannian_projection=riemannian_projection, random_state=random_state, initialization='random')
print("\n Eigenvalues calculated using numpy are :\n", p)
print("\n Eigenvalues calculate using the Eigengame are :\n", calc_eigengame_eigenvalues(X, Y, U1, V1))
print("\n Left Eigenvectors calculated using numpy are :\n", U1np)
print("\n Left Eigenvectors calculated using the Eigengame are :\n", U1)
print("\n Right Eigenvectors calculated using numpy are :\n", V1np)
print("\n Right Eigenvectors calculated using the Eigengame are :\n", V1)
print("\n Squared error in estimation of eigenvectors as compared to numpy :\n",
      np.sum((np.abs(U1np) - np.abs(U1)) ** 2, axis=0))